# Lesson 07: Three-way reconciliation (platform vs bridge vs LP)
**Level:** advanced

Every A-book trade exists in three places:
1. **Platform** deals: what the client was given (`data/platform/deals.csv`)
2. **Bridge** FIX ExecutionReports: what the LP told us in real time (`data/fix_logs`)
3. **LP statement**: what the LP says it executed, sent end of day (`data/lp_statements`)

If they disagree, the broker's hedge doesn't match its client positions, so it carries risk it doesn't know about.
This lesson matches them trade by trade, classifies every break and values the exposure. Expect the breaks from
**S05** (duplicate fill), **S06** (fill lost in the LP_C outage) and **S10** (gold quantity mapping).

In [1]:
import sys; sys.path.insert(0, "..")
import numpy as np, pandas as pd
from bridgelab import analysis as A
pd.set_option("display.width", 200, "display.max_columns", 30)

msgs = A.load_fix_messages()
quotes = A.lp_quotes(msgs)
orders, deals = A.load_platform()
lp_stmt = A.load_lp_statements()
sym, cfg, audit, clients = A.load_reference()
CONTRACT = sym.contract_size.to_dict()

platform = deals[deals.book == "A"].copy()
bridge = msgs[(msgs.msg_type == "8") & (msgs.exec_type == "F")].copy()
print(f"platform A-book deals: {len(platform)} | bridge fill reports: {len(bridge)} "
      f"(of which PossDup re-sends: {(bridge.poss_dup == 'Y').sum()}) | LP statement trades: {len(lp_stmt)}")

platform A-book deals: 745 | bridge fill reports: 745 (of which PossDup re-sends: 1) | LP statement trades: 745


## 1. Choose the matching key and the common unit
* **Key:** the LP's **ExecID** (tag 17). It's unique per execution, it's in the FIX message, the platform stores it
  (`lp_exec_id`), and the LP statement has it too. Never match on price/time alone.
* **Unit:** compare in **lots**, using the *official* contract size (100,000 for FX, **100 oz for gold**). Don't use
  the bridge's own conversion multiplier, because if that multiplier is wrong, it would hide the error.

In [2]:
bridge["lots"] = bridge.last_qty / bridge.symbol.map(CONTRACT)
lp_stmt["lots"] = lp_stmt.qty / lp_stmt.symbol.map(CONTRACT)
bridge_u = bridge[bridge.poss_dup != "Y"]            # one row per real execution

## 2. Recon 1: bridge (FIX) vs LP statement

In [3]:
r1 = bridge_u[["lp", "exec_id", "cl_ord_id", "symbol", "side", "lots", "last_px"]].merge(
    lp_stmt[["lp", "exec_id", "lots", "price", "trade_time"]], on=["lp", "exec_id"], how="outer",
    suffixes=("_bridge", "_lp"), indicator=True)
r1["status"] = r1._merge.astype(str).map({"both": "matched", "left_only": "missing at LP", "right_only": "missing at bridge"})
r1.loc[(r1.status == "matched") & ~np.isclose(r1.lots_bridge, r1.lots_lp), "status"] = "quantity differs"
r1.loc[(r1.status == "matched") & ~np.isclose(r1.last_px, r1.price), "status"] = "price differs"
r1.groupby(["lp", "status"]).size().unstack(fill_value=0)

status,matched,missing at bridge
lp,,
LP_A,389,0
LP_B,119,0
LP_C,236,1


In [4]:
r1[r1.status == "missing at bridge"]

,lp,exec_id,cl_ord_id,symbol,side,lots_bridge,last_px,lots_lp,price,trade_time,_merge,status
696,LP_C,LP_C-E000189,NaN,NaN,NaN,NaN,NaN,0.03,4096.8,2025-10-15 13:00:00.089,right_only,missing at bridge


**Break type 1: an execution the bridge never received (S06).** LP_C's statement shows a fill at ~13:00:00 that
isn't in our FIX log. The ExecutionReport was in flight when the session died, and the bridge reconnected with
**`141=Y` (reset sequence numbers)**, so LP_C never re-sent it. The platform rejected the client ("LP timeout"),
but LP_C really did trade.

Result: the broker holds a position at LP_C that **no client position offsets**, i.e. an unintended proprietary
position. Let's link it back to the client order:

In [5]:
lost = r1[r1.status == "missing at bridge"].merge(lp_stmt[["exec_id", "cl_ord_id", "side", "symbol"]], on="exec_id",
                                                  suffixes=("", "_stmt"))
lost["order_id"] = lost.cl_ord_id_stmt.str.extract(r"BR(\d+)-")[0].astype(int)
lost[["exec_id", "cl_ord_id_stmt", "symbol_stmt", "side_stmt", "lots_lp", "price", "trade_time"]].merge(
    orders[["order_id", "client_id", "lots", "status", "reject_reason"]], left_on=lost.order_id, right_on="order_id")

,exec_id,cl_ord_id_stmt,symbol_stmt,side_stmt,lots_lp,price,trade_time,order_id,client_id,lots,status,reject_reason
0,LP_C-E000189,BR701128-1,XAUUSD,SELL,0.03,4096.8,2025-10-15 13:00:00.089,701128,C1003,3.0,REJECTED,LP timeout


Note the LP quantity: it's a gold trade at LP_C, so the bridge sent ounces = lots × **1** (S10). The client asked for
3 lots, LP_C executed 3 oz = 0.03 lots. Two bugs in one trade.

## 3. Recon 2: platform deals vs bridge fills

In [6]:
r2 = platform[["deal_id", "order_id", "client_id", "lp", "lp_exec_id", "symbol", "side", "lots", "time"]].merge(
    bridge_u[["exec_id", "lots"]].rename(columns={"exec_id": "lp_exec_id", "lots": "lots_bridge"}),
    on="lp_exec_id", how="outer", indicator=True)
dupes = r2[r2.duplicated("lp_exec_id", keep=False) & r2.lp_exec_id.notna()]
print("platform deals sharing the same LP ExecID:")
dupes

platform deals sharing the same LP ExecID:


,deal_id,order_id,client_id,lp,lp_exec_id,symbol,side,lots,time,lots_bridge,_merge
463,901028,700965,C1001,LP_B,LP_B-E000075,EURUSD,BUY,2.0,2025-10-15 12:45:09.833,2.0,both
464,901029,700965,C1001,LP_B,LP_B-E000075,EURUSD,BUY,2.0,2025-10-15 12:45:10.016,2.0,both


**Break type 2: duplicate booking (S05).** One LP_B execution was booked as **two** platform deals: the original at
12:45:09 and a copy at 12:45:10, when LP_B re-sent the message with `43=PossDupFlag=Y` after the bridge's wrong
ResendRequest (Lesson 02). The client now has double the position they traded, and the broker has only half of
that hedged.

In [7]:
qty_breaks = r2[(r2._merge == "both") & ~np.isclose(r2.lots, r2.lots_bridge)]
print(f"platform vs bridge quantity mismatches: {len(qty_breaks)}")
qty_breaks.groupby(["lp", "symbol"]).agg(deals=("deal_id", "size"), platform_lots=("lots", "sum"),
                                         bridge_lots=("lots_bridge", "sum")).round(2)

platform vs bridge quantity mismatches: 97


,,deals,platform_lots,bridge_lots
lp,symbol,,,
LP_C,XAUUSD,97,85.21,0.85


**Break type 3: quantity mismatch (S10).** On every gold trade routed to LP_C, the platform booked N lots for the client
while the LP executed N **ounces** (1/100 of the size). The bridge's own conversion (`lp_quantity_multiplier
LP_C:XAUUSD = 1`) made both sides "agree" in real time, which is why nobody noticed. Only a reconciliation in
an independent unit exposes it.

## 4. Position reconciliation by LP and symbol
The final check: **net lots per LP and symbol** according to the platform (what we think we're hedged) vs according to
the LP statement (what we really hold).

In [8]:
sgn = lambda side: np.where(side == "BUY", 1, -1)
plat_pos = platform.assign(net=platform.lots * sgn(platform.side)).groupby(["lp", "symbol"]).net.sum()
lp_pos = lp_stmt.assign(net=lp_stmt.lots * sgn(lp_stmt.side)).groupby(["lp", "symbol"]).net.sum()
pos = pd.DataFrame({"platform_gross_lots": platform.groupby(["lp", "symbol"]).lots.sum(),
                    "lp_gross_lots": lp_stmt.groupby(["lp", "symbol"]).lots.sum(),
                    "platform_net_lots": plat_pos, "lp_net_lots": lp_pos}).fillna(0)
pos["break_lots"] = pos.platform_net_lots - pos.lp_net_lots
mid_last = {s: A.mid_series(quotes, s).iloc[-1] for s in sym.index}
pos["break_usd_notional"] = [b * CONTRACT[s] * mid_last[s] for (lp, s), b in zip(pos.index, pos.break_lots)]
pos.round(2)

platform_gross_lots  lp_gross_lots  platform_net_lots  lp_net_lots  break_lots  break_usd_notional
lp   symbol                                                                                                    
LP_A EURUSD               375.14         375.14             -48.32       -48.32        0.00                0.00
     GBPUSD                81.45          81.45              14.35        14.35        0.00                0.00
     XAUUSD               307.14         307.14              42.78        42.78        0.00                0.00
LP_B EURUSD                67.19          65.19              14.37        12.37        2.00           233074.00
     GBPUSD                17.05          17.05              -6.77        -6.77        0.00                0.00
     XAUUSD               113.19         113.19              -9.31        -9.31       -0.00               -0.00
LP_C EURUSD               195.01         195.01              32.69        32.69        0.00                0.00
     GBPUSD                79.15          79.15             -10.67       -10.67        0.00                0.00
     XAUUSD                85.21           0.88              -2.13        -0.05       -2.08          -850818.79

Reading the breaks:
* **LP_C / XAUUSD:** the **gross** volume shows the scale of S10: the platform booked ~85 lots of gold against LP_C
  while LP_C executed under 1 lot. Clients bought and sold, so most of it netted out, but the remaining **net**
  break is still about 2 lots, roughly **$850k of gold** the broker thinks is hedged and isn't. If gold moves $10,
  that's about $2,000 of P&L nobody decided to take, and it changes with every new LP_C gold trade.
* **LP_B / EURUSD:** the duplicate deal (S05). The platform shows 2 lots more than LP_B executed.
* **LP_C / XAUUSD** also contains the lost fill from the outage (S06), in the *opposite* direction (the LP has
  a position the platform doesn't).

## 5. Break register

In [9]:
register = pd.DataFrame([
    dict(break_id="RB-1", scenario="S06", type="Execution missing at bridge/platform",
         where="LP_C statement vs FIX", detail=f"{len(lost)} LP_C fill(s) at ~13:00 never received; client order rejected 'LP timeout'",
         root_cause="Session died with the ExecutionReport in flight; reconnect used ResetSeqNumFlag=Y, so no resend",
         action="Close or transfer the orphan LP position; never reset sequence numbers after an unclean disconnect; "
                "treat timeouts as 'unknown' and query order status (35=H) on reconnect"),
    dict(break_id="RB-2", scenario="S05", type="Duplicate deal", where="platform vs FIX",
         detail=f"ExecID {dupes.lp_exec_id.iloc[0]} booked {len(dupes)} times",
         root_cause="ResendRequest from the wrong sequence number + PossDup (43=Y) not de-duplicated on ExecID",
         action="Cancel the duplicate client deal (with client notice); de-duplicate on ExecID; fix BeginSeqNo"),
    dict(break_id="RB-3", scenario="S10", type="Quantity mismatch", where="platform vs FIX vs LP (in lots)",
         detail=f"{len(qty_breaks)} LP_C gold trades executed in ounces = lots x 1",
         root_cause="lp_quantity_multiplier LP_C:XAUUSD = 1 set at onboarding (CHG-2291)",
         action="Hedge the open gap now; fix the multiplier to 100; pre-trade check qty/lots = contract size; "
                "daily position recon per LP"),
])
register

,break_id,scenario,type,where,detail,root_cause,action
0,RB-1,S06,Execution missing at bridge/platform,LP_C statement vs FIX,1 LP_C fill(s) at ~13:00 never received; clien...,Session died with the ExecutionReport in fligh...,Close or transfer the orphan LP position; neve...
1,RB-2,S05,Duplicate deal,platform vs FIX,ExecID LP_B-E000075 booked 2 times,ResendRequest from the wrong sequence number +...,Cancel the duplicate client deal (with client ...
2,RB-3,S10,Quantity mismatch,platform vs FIX vs LP (in lots),97 LP_C gold trades executed in ounces = lots x 1,lp_quantity_multiplier LP_C:XAUUSD = 1 set at ...,Hedge the open gap now; fix the multiplier to ...


## Reconciliation checklist (daily)
1. Match **bridge fills ↔ LP statement** on ExecID → missing/extra executions, price and quantity differences.
2. Match **platform deals ↔ bridge fills** on ExecID → duplicates, orphans, quantity conversions.
3. Compare **net positions per LP and symbol** in an independent unit (lots via the official contract size).
4. Value every break at the current price and escalate above a threshold.
5. Investigate unanswered orders (timeouts) with the LP **the same day**: they are unknowns, not rejects.

**Next:** Lesson 08 turns every check from Lessons 02–07 into an automated monitor and incident playbook.